In [43]:
# importing libraries

In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

In [45]:
df = pd.read_csv("real_estate_listings_india_2025.csv")
print("Original Shape:", df.shape)

Original Shape: (287, 9)


In [46]:
def convert_size(size):
    try:
        size = str(size).replace(" sqft", "").strip()
        if "-" in size:
            low, high = size.split("-")
            return (float(low) + float(high)) / 2
        else:
            return float(size)
    except:
        return np.nan

df["size_avg"] = df["size"].apply(convert_size)

In [47]:
# clean data

In [48]:
df.drop(columns=["url", "date", "size"], inplace=True, errors="ignore")
df.dropna(inplace=True)
print("Cleaned Shape:", df.shape)

Cleaned Shape: (226, 7)


In [49]:
# Log Transform Target

In [50]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df[df["price"] > 0]
df["log_price"] = np.log(df["price"])

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["log_price"])

X = df.drop(columns=["price", "log_price"])
y = df["log_price"]

In [51]:
# Identify Categorical Columns

In [52]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['city', 'type', 'neighborhood']
Numeric: ['beds', 'baths', 'size_avg']


In [53]:
# Preprocessing (One-Hot Encoding)

In [54]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

In [55]:
# Define Models

In [56]:
xgb_model = XGBRegressor(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgbm_model = LGBMRegressor(
    n_estimators=150,
    max_depth=3,
    learning_rate=0.05,
    num_leaves=15,
    verbose=-1,
    random_state=42
)

In [57]:
# create pipeline

In [58]:
xgb_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", xgb_model)
])

lgbm_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", lgbm_model)
])

In [59]:
# cross validation

In [60]:
kf = KFold(n_splits=10, shuffle=True, random_state=42)
def evaluate_model(pipeline, X, y):
    mae_scores = []
    rmse_scores = []
    r2_scores = []

    for train_index, test_index in kf.split(X):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        pipeline.fit(X_train, y_train)
        preds = pipeline.predict(X_test)

        y_test_exp = np.exp(y_test)
        preds_exp = np.exp(preds)

        mae = mean_absolute_error(y_test_exp, preds_exp)
        rmse = np.sqrt(mean_squared_error(y_test_exp, preds_exp))
        r2 = r2_score(y_test_exp, preds_exp)

        mae_scores.append(mae)
        rmse_scores.append(rmse)
        r2_scores.append(r2)

    return {
        "MAE": np.mean(mae_scores),
        "RMSE": np.mean(rmse_scores),
        "R2": np.mean(r2_scores)
    }

print("Evaluating XGBoost...")
xgb_results = evaluate_model(xgb_pipeline, X, y)

print("Evaluating LightGBM...")
lgbm_results = evaluate_model(lgbm_pipeline, X, y)

Evaluating XGBoost...
Evaluating LightGBM...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

In [61]:
# Train Final Model on Full Data

In [62]:
if xgb_results["MAE"] < lgbm_results["MAE"]:
    final_model = xgb_pipeline
    best_model_name = "XGBoost"
else:
    final_model = lgbm_pipeline
    best_model_name = "LightGBM"

print(f"Best Model Selected: {best_model_name}")

Best Model Selected: XGBoost


In [63]:
# User input example

In [65]:
new_data = pd.DataFrame({
    "beds": [2],
    "baths": [2],
    "size_avg": [1200],
    "city": ["Hyderabad"],
    "type": ["2 BHK Flat"],
    "neighborhood": ["Madhapur"]
})

log_pred = final_model.predict(new_data)
price_pred = np.exp(log_pred)

print("Predicted Price: ₹{:,.0f}".format(price_pred[0]))

Predicted Price: ₹7,456,185
